### Part 1 - ai_tasks x topics

In [2]:
import json
from tqdm import tqdm

In [ ]:
# Function to parse the string and create a JSON object
def parse_gpt_response(string, id, task, topic):
    """
    Helper function fur parse_results()
    Parses the gpt responses from string to dict
    """
# Create a dictionary to hold the data in the desired format
# Only successfull if all that is expected is present

    try:
        # Attempt to load the JSON string
        # Decode any UTF-8 character codes in the input string
        decoded_string = (string.encode().decode('unicode_escape')).encode('latin1').decode('utf-8')

        data_dict = {}
        data_dict["id"] = id
        data_dict["prompt"] = decoded_string
        data_dict["context"] = {"task": task, "topic": topic}
        return data_dict

    except json.JSONDecodeError as e:
        return None

In [4]:
path_to_results = "../results/gpt_results.jsonl"
output_path = "../results/parsed_prompts_tasks_x_topics.json"


# TODO: do both in the same for loop, no need to store contents
# Extract content from json
print("Parsing GPT responses...")
medical_prompts = []
questions_failed_to_parse = []
with open(path_to_results, 'r') as file:
    for line in tqdm(file):
        try:
            data = json.loads(line)  # Parse each line as JSON
            response_content = data.get("response", {}).get("body", {}).get("choices", [])[0].get("message", {}).get("content", None)
            id = data.get("custom_id") # format "0-0" "task_id - subtopic_id"
            if response_content and id:
                parsed_response = parse_gpt_response(response_content, int(id.split("-")[0]), str(id.split("-")[1]), str(id.split("-")[2]))
                if bool(parsed_response):
                    medical_prompts.append(parsed_response)
                else:
                    questions_failed_to_parse.append(id)
        except json.JSONDecodeError as e:
            questions_failed_to_parse.append(id)
print("Parsing completed")

# Save to JSON file
with open(output_path, 'w') as json_file:
    json.dump(medical_prompts, json_file, indent=4)

# Output the parsed data (for verification)
print(f"Medical prompts have been saved to {output_path}")
print("See an example below:")
print(json.dumps(parsed_response, indent=4))
print(f"Failed to parse {len(questions_failed_to_parse)} questions:")
print(questions_failed_to_parse)

Parsing GPT responses...


637it [00:00, 31424.84it/s]

Parsing completed
Medical prompts have been saved to ../results/parsed_prompts_tasks_x_topics.json
See an example below:
{
    "id": 636,
    "prompt": "As a primary care physician looking to enhance preventive care services in my practice, I am reviewing new regulations regarding the integration of routine vaccination programs for adults aged 50 and older, including the updated recommendations for pneumococcal and Zoster vaccines. 1. Can you provide an overview of the latest regulatory guidelines on these vaccines and outline any policy changes compared to previous years? 2. What strategies can I implement to ensure compliance with these guidelines while also improving patient adherence to vaccination schedules? 3. How can electronic health record systems be utilized to track compliance and generate reminders for both patients and staff regarding upcoming vaccines? 4. Are there any specific resources or incentives provided by health agencies or insurers that facilitate the adoption of